<a href="https://colab.research.google.com/github/Ololade117/Inducing-Polysemanticity/blob/main/The__flexible_bias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The Bias in simple terms offsets the curve in a way the Weight matrix cant. The goal of this research is to derive an approach towards having a more flexible bias(not just one value) that will be determined by the input, that can nudge the learning or prediction into the right direction easily with a lesser number of parameters(< weight matrix dim). The idea is to make a bias that does not contain just one number but several which could offset the learning better

The task:
1. Derive a means to make the weight matrix more flexible. I.e before y= wX +b , b  is obtained from B that contains different bs, b is most suited for X
2. Compare this with a model of similar paramter size

Hypothesis:
A model with a flexible bias(list of several biases)  will out perform a model with a fixed bias(one number)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

In [ ]:
with open('names.txt', 'r') as f:
    names = f.read().splitlines()

# Add a special start-of-sequence token and end-of-sequence token
# Using '.' for EOS and '^' for SOS for simplicity, assuming they are not in names
names = [f'^{name}.' for name in names]

# Build character vocabulary
chars = sorted(list(set(''.join(names))))
vocab_size = len(chars)

char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

print(f'Vocabulary size: {vocab_size}')
print(f'Example names: {names[:5]}')

Vocabulary size: 28
Example names: ['^emma.', '^olivia.', '^ava.', '^isabella.', '^sophia.']


In [ ]:
# Create training data (input, target pairs)
X = []
y = []

for name in names:
    for i in range(len(name) - 1):
        input_char = name[i]
        target_char = name[i+1]
        X.append(char_to_ix[input_char])
        y.append(char_to_ix[target_char])

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

print(f'Total training examples: {len(X)}')
print(f'First 5 input indices: {X[:5]}')
print(f'First 5 target indices: {y[:5]}')

Total training examples: 228146
First 5 input indices: tensor([ 1,  6, 14, 14,  2])
First 5 target indices: tensor([ 6, 14, 14,  2,  0])


# Normal Bias

In [ ]:
class ManualFeedforwardNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=1):
        super(ManualFeedforwardNet, self).__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Manually create embedding table (C)
        self.C = torch.randn((vocab_size, embedding_dim), requires_grad=True)

        # Manually create weights/biases for each hidden layer
        self.Ws = []
        self.bs = []
        in_dim = embedding_dim
        for i in range(num_layers):
            W = torch.randn((in_dim, hidden_dim), requires_grad=True)
            b = torch.randn(hidden_dim, requires_grad=True)
            self.Ws.append(W)
            self.bs.append(b)
            in_dim = hidden_dim  # next layer's input is this layer's hidden_dim

        # Output layer weights/biases
        self.W_out = torch.randn((hidden_dim, vocab_size), requires_grad=True)
        self.b_out = torch.randn(vocab_size, requires_grad=True)

        # Store parameters in a list for the optimizer
        self.parameters_list = [self.C] + self.Ws + self.bs + [self.W_out, self.b_out]

    def forward(self, x):
        # x is a tensor of character indices
        # Embedding lookup
        emb = self.C[x]  # (batch_size, embedding_dim)

        # Pass through each hidden layer manually — edit this loop freely
        h = emb
        for W, b in zip(self.Ws, self.bs):
            h = torch.tanh(h @ W + b)

        # Output layer
        logits = h @ self.W_out + self.b_out  # (batch_size, vocab_size)
        return logits


# Model parameters
embedding_dim = 10
hidden_dim = 128
num_layers = 2  # <-- set however many hidden layers you want

# Instantiate the new model
manual_model = ManualFeedforwardNet(vocab_size, embedding_dim, hidden_dim, num_layers)

# Define loss function and optimizer for the manual model
criterion_manual = nn.CrossEntropyLoss()
optimizer_manual = optim.Adam(manual_model.parameters_list, lr=0.01)

print(manual_model)
print(f'Total number of manually created parameters: {sum(p.numel() for p in manual_model.parameters_list)}')

# Assign the manual model to 'model' so subsequent training cells can use it
normal_model1 = manual_model
criterion = criterion_manual
optimizer = optimizer_manual